In [0]:
%sql
DROP TABLE IF EXISTS bronze_entregas;
DROP TABLE IF EXISTS bronze_comercial;
DROP TABLE IF EXISTS bronze_vendedores;
DROP TABLE IF EXISTS bronze_produtos;
DROP TABLE IF EXISTS bronze_ocorrencias;
DROP TABLE IF EXISTS bronze_pedidos;
DROP TABLE IF EXISTS bronze_itens;
DROP TABLE IF EXISTS bronze_regioes;
DROP TABLE IF EXISTS bronze_logistica;
DROP TABLE IF EXISTS bronze_clientes;

In [0]:

from pyspark.sql.functions import (
    col,
    current_timestamp,
    sha2,
    concat_ws,
    to_json,
    lit
)
from pyspark.sql.types import StructType, ArrayType, MapType

BASE_PATH = "/Volumes/workspace/default/sources"


# ==========================================
# CSV SMART READER
# ==========================================
def read_csv_smart(path):
    delimiters = [",", ";", "|", "\t"]

    for sep in delimiters:
        try:
            df = spark.read \
                .option("header", True) \
                .option("inferSchema", True) \
                .option("sep", sep) \
                .option("mode", "PERMISSIVE") \
                .csv(path)

            if len(df.columns) > 1:
                print(f"✔ Delimitador detectado: {sep} -> {path}")
                return df

        except Exception:
            continue

    raise Exception(f"❌ Não foi possível identificar delimitador: {path}")


# ==========================================
# BRONZE ENRICH (100% SAFE + SERVERLESS OK)
# ==========================================
def enrich_bronze(df, source_name):

    safe_cols = []

    for f in df.schema.fields:
        c = col(f.name)

        # STRUCT / ARRAY / MAP → JSON STRING
        if isinstance(f.dataType, (StructType, ArrayType, MapType)):
            safe_cols.append(to_json(c).alias(f.name))
        else:
            safe_cols.append(c.cast("string").alias(f.name))

    df_safe = df.select(*safe_cols)

    # 🔥 HASH totalmente seguro (somente STRING)
    hash_cols = concat_ws("||", *df_safe.columns)

    return df_safe \
        .withColumn("_source_system", lit(source_name)) \
        .withColumn("_ingestion_timestamp", current_timestamp()) \
        .withColumn("_raw_hash", sha2(hash_cols, 256))


# ==========================================
# SAVE DELTA (SERVERLESS SAFE)
# ==========================================
def save_delta(df, table_name):
    path = f"{BASE_PATH}/delta/{table_name}"

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(path)

    print(f"✔ Delta salvo em: {path}")


# ==========================================
# CLIENTES
# ==========================================
clientes = enrich_bronze(
    read_csv_smart(f"{BASE_PATH}/crm_clientes_export.csv"),
    "clientes"
)
save_delta(clientes, "bronze_clientes")


# ==========================================
# PRODUTOS
# ==========================================
produtos = enrich_bronze(
    spark.read.option("multiline", True)
        .json(f"{BASE_PATH}/cadastro_produtos_api_dump.json"),
    "produtos"
)
save_delta(produtos, "bronze_produtos")


# ==========================================
# PEDIDOS CABEÇALHO
# ==========================================
pedidos = enrich_bronze(
    read_csv_smart(f"{BASE_PATH}/erp_pedidos_cabecalho_2025.csv"),
    "pedidos"
)
save_delta(pedidos, "bronze_pedidos")


# ==========================================
# ITENS
# ==========================================
itens = enrich_bronze(
    read_csv_smart(f"{BASE_PATH}/erp_pedidos_itens_2025.csv"),
    "itens"
)
save_delta(itens, "bronze_itens")


# ==========================================
# ENTREGAS
# ==========================================
entregas = enrich_bronze(
    spark.read.option("multiline", True)
        .json(f"{BASE_PATH}/logistica_entregas.json"),
    "entregas"
)
save_delta(entregas, "bronze_entregas")


# ==========================================
# OCORRÊNCIAS
# ==========================================
ocorrencias = enrich_bronze(
    spark.read.json(f"{BASE_PATH}/atendimento_ocorrencias.ndjson"),
    "ocorrencias"
)
save_delta(ocorrencias, "bronze_ocorrencias")


# ==========================================
# CANAIS
# ==========================================
canais = enrich_bronze(
    read_csv_smart(f"{BASE_PATH}/comercial_canais.csv"),
    "canais"
)
save_delta(canais, "bronze_canais")


# ==========================================
# REGIÕES
# ==========================================
regioes = enrich_bronze(
    spark.read \
        .option("header", True) \
        .option("inferSchema", True) \
        .option("sep", "|") \
        .csv(f"{BASE_PATH}/legado_regioes_pipe.txt"),
    "regioes"
)
save_delta(regioes, "bronze_regioes")


# ==========================================
# VENDEDORES
# ==========================================
vendedores = enrich_bronze(
    read_csv_smart(f"{BASE_PATH}/vendedores.csv"),
    "vendedores"
)
save_delta(vendedores, "bronze_vendedores")


print("✔ BRONZE LAYER FINAL (SERVERLESS SAFE) CRIADO COM SUCESSO")


✔ Delimitador detectado: ; -> /Volumes/workspace/default/sources/crm_clientes_export.csv
✔ Delta salvo em: /Volumes/workspace/default/sources/delta/bronze_clientes
✔ Delta salvo em: /Volumes/workspace/default/sources/delta/bronze_produtos
✔ Delimitador detectado: ; -> /Volumes/workspace/default/sources/erp_pedidos_cabecalho_2025.csv
✔ Delta salvo em: /Volumes/workspace/default/sources/delta/bronze_pedidos
✔ Delimitador detectado: , -> /Volumes/workspace/default/sources/erp_pedidos_itens_2025.csv
✔ Delta salvo em: /Volumes/workspace/default/sources/delta/bronze_itens
✔ Delta salvo em: /Volumes/workspace/default/sources/delta/bronze_entregas
✔ Delta salvo em: /Volumes/workspace/default/sources/delta/bronze_ocorrencias
✔ Delimitador detectado: ; -> /Volumes/workspace/default/sources/comercial_canais.csv
✔ Delta salvo em: /Volumes/workspace/default/sources/delta/bronze_canais
✔ Delta salvo em: /Volumes/workspace/default/sources/delta/bronze_regioes
✔ Delimitador detectado: ; -> /Volumes/wo